# Czech800 — Google Colab GPU Speech-to-Text (timestamp + resume)

**Source:** Private Google Drive `Czech800_tairsan.mkv` — file ID `1z_5kOoDJyUrxWqDkd3QdxKzOkdsW0X3F` (1,265,839,563 bytes in Drive metadata). 

**Output:** Czech/English speech transcription with *absolute* segment and word start/end times, `SRT`, `CSV`, `JSONL`, `TXT` in `MyDrive/Czech800_STT_6h`. Saves one checkpoint per 180-second chunk so interrupted runs can resume. Timestamps are estimates, not proven ±1 ms. **Does not cut audio, verify speech boundaries, or add Mongolian translations.**

**Instructions (MN):** Colab → Runtime → Change runtime type → **T4 GPU** (эсвэл өөр NVIDIA GPU). Дараа нь **Runtime → Run all**. Google-ийн зөвшөөрөл асуувал файлын эзэн `zazadada8899@gmail.com`-ээр нэвтэр. Эхний удаа эх MKV-г Colab-ийн түр диск рүү татна. Colab тасарвал notebook-ийг дахин ажиллуулахад Drive дахь дууссан `chunk_*.json` хэсгүүдийг алгасна. Компьютер дээр байгаа файлыг дахин татах шаардлагагүй.

**Important:** Source is private. Do not set Drive sharing to public. This notebook runs in your signed-in Colab runtime; ChatGPT cannot start or monitor that runtime from here.


In [ ]:
!pip -q install --upgrade faster-whisper google-api-python-client google-auth
!ffmpeg -version | head -n 1
!nvidia-smi


## 1. Эх файлын ID, хадгалах хавтас, загвар

`MODEL_NAME="large-v3"` — Чех–Англи холимог ярианы чанарт илүү анхаарсан горим. Хурд чухал бол `"medium"` болгож болно; нэг багц дундаас загвар өөрчилбөл үр дүн холилдоно, тиймээс тусдаа output хавтас ашиглаарай. `MAX_CHUNKS=0` бүх хэсгийг боловсруулна. Түр тестэд `MAX_CHUNKS=2` тавьж болно.


In [ ]:
from pathlib import Path
import os, json, csv, subprocess, math, gc, shutil, time, re
from datetime import datetime, timezone

FILE_ID = "1z_5kOoDJyUrxWqDkd3QdxKzOkdsW0X3F"
EXPECTED_NAME = "Czech800_tairsan.mkv"
OUTPUT_DIR = Path("/content/drive/MyDrive/Czech800_STT_6h")
LOCAL_VIDEO = Path("/content/Czech800_tairsan.mkv")
MODEL_NAME = "large-v3"  # for speed, switch to 'medium' BEFORE FIRST RUN
CHUNK_SEC = 180
OVERLAP_SEC = 10
MAX_CHUNKS = 0  # 0 = all remaining; 2 = two-chunk pilot
BEAM_SIZE = 5

assert 0 <= OVERLAP_SEC < CHUNK_SEC
STEP_SEC = CHUNK_SEC - OVERLAP_SEC
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "chunks").mkdir(exist_ok=True)
print("Drive checkpoint directory:", OUTPUT_DIR)


## 2. Google Drive нээх, эх MKV-г файл ID-аар татах

Drive-д холбогдсон Google аккаунт видео файлыг харах эрхтэй байх ёстой. Google Drive API-гаар **private** видеог татна — нийтийн линк болгох шаардлагагүй. Colab runtime дахин эхэлбэл түр диск дээрх эх файл алга болдог, гэхдээ Drive-д хадгалсан transcript хэсгүүд үлдэнэ.


In [ ]:
from google.colab import drive, auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

drive.mount("/content/drive")
auth.authenticate_user()
creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
service = build("drive", "v3", credentials=creds, cache_discovery=False)
meta = service.files().get(fileId=FILE_ID,fields="id,name,size,mimeType").execute()
expected_bytes = int(meta["size"])
print("Drive media:", meta)
if LOCAL_VIDEO.exists() and LOCAL_VIDEO.stat().st_size == expected_bytes:
    print("Existing local media: valid size, skip download")
else:
    if LOCAL_VIDEO.exists():
        LOCAL_VIDEO.unlink()
    target = LOCAL_VIDEO.with_suffix(".partial")
    if target.exists():
        target.unlink()  # partial file is not a valid video
    request = service.files().get_media(fileId=FILE_ID)
    with open(target, "wb") as fh:
        dl = MediaIoBaseDownload(fh, request, chunksize=32*1024*1024)
        done = False
        while not done:
            status, done = dl.next_chunk(num_retries=5)
            if status:
                print(f"Downloading: {status.progress()*100:5.1f}%", end="\r")
    assert target.stat().st_size == expected_bytes, "Incomplete Drive download"
    target.replace(LOCAL_VIDEO)
    print("\nDownload done")

probe = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration:stream=codec_type,codec_name",
                        "-of","json",str(LOCAL_VIDEO)],capture_output=True,text=True,check=True)
media_info=json.loads(probe.stdout)
DURATION=float(media_info["format"]["duration"])
assert any(s.get("codec_type")=="audio" for s in media_info["streams"]), "No audio stream in MKV"
print("Duration:",DURATION,"seconds =",round(DURATION/3600,3),"hours")
print("Codecs:",media_info["streams"])


## 3. NVIDIA GPU дээр Faster-Whisper ажиллуулах

`multilingual=True` нь ярианы цонх бүрт хэл дахин таамаглана. Энэ нь Чех–Англи холимог контентод тус болох ч өгүүлбэр бүрийг 100% зөв ялгахгүй, мөн сегмент болгонд `language` гэсэн баталгаатай талбар гаргахгүй. `word_timestamps=True` нь үг тус бүрийн **тооцоолсон** хугацаа өгнө. <small>Check the console for CUDA/cuDNN compatibility errors.</small>


In [ ]:
import ctranslate2
from faster_whisper import WhisperModel

gpus = ctranslate2.get_cuda_device_count()
print("CUDA GPUs detected:",gpus)
assert gpus > 0, "No NVIDIA GPU. Colab Runtime > Change runtime type > GPU."
try:
    model = WhisperModel(MODEL_NAME,device="cuda",compute_type="float16",download_root="/content/whisper_models")
except Exception as e:
    raise RuntimeError(f"Failed to load GPU model: {e}. Check Colab CUDA/cuDNN and Runtime GPU selection.") from e
print("Whisper GPU model loaded:", MODEL_NAME)


## 4. 6 цагийн яриаг хэсэгчлэн таних — тасарсан ч үргэлжилнэ

**Absolute timestamps:** Хэсэг бүр эх бичлэг дэх хугацаандаа буцаж нэмэгдэнэ. 180 секундийн цонхыг 10 секунд давхарлуулж ажиллуулж, давхардлыг төв цэгээр нь шүүнэ. Тасарсан хэсгүүдэд дууны эхний/сүүлийн авиа алдагдах эрсдэл үлдэх тул boundary QA (хил заагийн шалгалт) шаардлагатай.

**Checkpoint:** `MyDrive/Czech800_STT_6h/chunks/chunk_00000.json`, ... дууссан хэсэг бүрийг Drive-д хадгална. Drive салсан бол ажил зогсоно — дахин нэвтрээд үргэлжлүүлэх боломжтой.


In [ ]:
def timestamp_srt(seconds):
    ms=max(0,round(float(seconds)*1000))
    h,ms=divmod(ms,3600000)
    m,ms=divmod(ms,60000)
    s,ms=divmod(ms,1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def atomic_text(path,text):
    path=Path(path)
    temp=path.with_name(path.name+".tmp")
    temp.write_text(text,encoding="utf-8")
    os.replace(temp,path)

def gather():
    full=[]
    for p in sorted((OUTPUT_DIR/"chunks").glob("chunk_*.json")):
        try:
            obj=json.loads(p.read_text(encoding="utf-8"))
            if obj.get("model") != MODEL_NAME or obj.get("file_id") != FILE_ID:
                raise RuntimeError(f"Settings differ: {p}. Use a fresh OUTPUT_DIR.")
            full.extend(obj["segments"])
        except Exception as e:
            print("Bad checkpoint:",p,e)
            raise
    return sorted(full,key=lambda r:(r["start_sec"],r["end_sec"]))

def export_all():
    records=gather()
    sub=[]
    for i,s in enumerate(records,1):
        sub.append(f"{i}\n{timestamp_srt(s['start_sec'])} --> {timestamp_srt(s['end_sec'])}\n{s['text'].strip()}\n")
    atomic_text(OUTPUT_DIR/"transcript.srt","\n".join(sub))
    atomic_text(OUTPUT_DIR/"transcript.txt","\n".join(f"[{timestamp_srt(s['start_sec'])} --> {timestamp_srt(s['end_sec'])}] {s['text']}" for s in records))
    atomic_text(OUTPUT_DIR/"segments.jsonl","\n".join(json.dumps(s,ensure_ascii=False) for s in records)+"\n")
    import io
    out=io.StringIO();wr=csv.writer(out)
    wr.writerow(["segment","start_sec","end_sec","start_ms","end_ms","text","avg_logprob","no_speech_prob"])
    for i,s in enumerate(records,1):
        wr.writerow([i,s["start_sec"],s["end_sec"],s["start_ms"],s["end_ms"],s["text"],s["avg_logprob"],s["no_speech_prob"]])
    atomic_text(OUTPUT_DIR/"segments.csv",out.getvalue())
    out=io.StringIO();wr=csv.writer(out)
    wr.writerow(["segment","word_index","start_sec","end_sec","start_ms","end_ms","word","probability"])
    for i,s in enumerate(records,1):
        for j,w in enumerate(s["words"],1):
            wr.writerow([i,j,w["start_sec"],w["end_sec"],w["start_ms"],w["end_ms"],w["word"],w["probability"]])
    atomic_text(OUTPUT_DIR/"words.csv",out.getvalue())
    print("EXPORTED:",len(records),"segments →",OUTPUT_DIR)

# On a resume, existing chunks with the same file ID+model are preserved.
N=math.ceil(max(0,DURATION-CHUNK_SEC)/STEP_SEC)+1
if DURATION <= CHUNK_SEC: N=1
processed_now=0
start_wall=time.time()
for index in range(N):
    if MAX_CHUNKS and processed_now >= MAX_CHUNKS:
        print("Pilot limit reached: MAX_CHUNKS =",MAX_CHUNKS)
        break
    begin=index*STEP_SEC
    if begin>=DURATION: break
    length=min(CHUNK_SEC,DURATION-begin)
    cp=OUTPUT_DIR/"chunks"/f"chunk_{index:05d}.json"
    if cp.exists():
        previous=json.loads(cp.read_text(encoding="utf-8"))
        assert previous.get("file_id")==FILE_ID and previous.get("model")==MODEL_NAME and previous.get("chunk_sec")==CHUNK_SEC, f"Incompatible checkpoint: {cp}"
        print(f"SKIP completed {index+1}/{N}: {begin:.1f}s")
        continue
    wav=Path(f"/content/clip_{index:05d}.wav")
    try:
        subprocess.run(["ffmpeg","-nostdin","-v","error","-y","-ss",f"{begin:.3f}","-i",str(LOCAL_VIDEO),
                        "-t",f"{length:.3f}","-vn","-ac","1","-ar","16000","-c:a","pcm_s16le",str(wav)],check=True)
        gen,info=model.transcribe(str(wav),language=None,multilingual=True,word_timestamps=True,
                                  vad_filter=True,condition_on_previous_text=False,beam_size=BEAM_SIZE)
        output=[]
        keep_until=min(DURATION,begin+STEP_SEC)
        for s in gen:  # generator MUST be iterated to perform inference
            absolute_start=begin+float(s.start)
            absolute_end=begin+float(s.end)
            midpoint=(absolute_start+absolute_end)/2
            if not (begin<=midpoint<keep_until):
                continue  # prevent repeated lines in overlap
            words=[]
            for w in (s.words or []):
                a=begin+float(w.start);b=begin+float(w.end)
                words.append({"word":w.word,"start_sec":round(a,3),"end_sec":round(b,3),
                              "start_ms":round(a*1000),"end_ms":round(b*1000),
                              "probability":round(float(w.probability),4)})
            output.append({"text":s.text.strip(),"start_sec":round(absolute_start,3),
                           "end_sec":round(absolute_end,3),
                           "start_ms":round(absolute_start*1000),"end_ms":round(absolute_end*1000),
                           "avg_logprob":round(float(s.avg_logprob),4),
                           "no_speech_prob":round(float(s.no_speech_prob),4),
                           "chunk_index":index,"words":words})
        meta={"file_id":FILE_ID,"model":MODEL_NAME,"chunk_sec":CHUNK_SEC,
              "overlap_sec":OVERLAP_SEC,"index":index,"begin_sec":round(begin,3),
              "length_sec":round(length,3),"detected_language_initial":info.language,
              "status":"ASR_ESTIMATE_NOT_VERIFIED","segments":output,
              "finished_utc":datetime.now(timezone.utc).isoformat()}
        atomic_text(cp,json.dumps(meta,ensure_ascii=False,indent=2))
        processed_now+=1
        print(f"DONE {index+1}/{N}: {begin:.1f}s..{begin+length:.1f}s; {len(output)} segments; elapsed {int(time.time()-start_wall)}s",flush=True)
        if processed_now % 5 == 0:
            export_all()
    finally:
        wav.unlink(missing_ok=True)
    gc.collect()

export_all()
print("END: existing chunks:",len(list((OUTPUT_DIR/"chunks").glob("chunk_*.json"))),"/",N)


## 5. Дууссан эсэхийг шалгах / Drive-аас авах

Google Drive → My Drive → `Czech800_STT_6h` → `transcript.srt`, `transcript.txt`, `segments.csv`, `words.csv`, `segments.jsonl`.

- `transcript.srt`: видео дээр хадмал болгон харуулах **өгүүлбэрийн хугацаа**.
- `words.csv`: үг бүрийн `start_sec,end_sec,start_ms,end_ms` **урьдчилсан хугацаа**.
- `chunks/`: тасарвал үргэлжлүүлэх хадгалсан хэсгүүд.
- **Аудио таслахаас өмнө** Чех/Англи дуудлага, өгүүлбэрийн харьяалал, хилийг видео + waveform + Forced Alignment-р нягтлах хэрэгтэй.

**Тасарвал:** Colab-ыг дахин холбоод бүх cell-ийг ажиллуул. Эх 1.27GB MKV-г Colab-ийн түр диск рүү дахин татаж магадгүй, харин дууссан сегментүүд Drive-д хэвээр тул давтан танихгүй.
